In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import re
import glob 

On associe une province à chaque Ping en bouclant sur les fichier provinces

In [ ]:
ds_sv_path = Path("/data/rd_exchange/salbernhe/data/acoustics/profiles38kHz10t0750m20251006.nc")


ds_sv = xr.open_dataset(ds_sv_path)
ds_sv = ds_sv.assign_coords(
    year=("time", ds_sv["time"].dt.year.values),
    month=("time", ds_sv["time"].dt.month.values)

)

idx_ping_prov = np.full(ds_sv.dims["time"], np.nan, dtype=object)

prov_folder = sorted(glob.glob("/data/rd_exchange/asauvebois/moy_month/*_monthly_mean.nc"))

In [ ]:
# On boucle sur les fichies provinces

for file_prov in prov_folder:
    nom = file_prov.split("/")[-1].replace(".nc", "")   # "2003_12_monthly_mean"

    year_str = int(file_prov.split("_")[0])
    month_str = int(file_prov.split("_")[1])

    mask= (ds_sv["year"]==year_str) & (ds_sv["month"]==month_str)
    if not mask.any():
        continue
   
    # on va chercher le point de grille le plus proche 

    sub = ds_sv.sel(time=mask) #on applique le mask qui filtre le ds_sv et ne garde que les ping du mois/année en cours
    ds_prov = xr.open_dataset(file_prov)




In [ ]:
import xarray as xr
import numpy as np
import glob
import re

# --- 1. Charger et préparer les données Sv ---
ds_sv = xr.open_dataset("sv_data.nc")
ds_sv = ds_sv.assign_coords(
    year=("time", ds_sv["time"].dt.year.values),
    month=("time", ds_sv["time"].dt.month.values)
)

province_values = np.full(ds_sv.dims["time"], np.nan, dtype=object)

# --- 2. Lister tous les fichiers province du dossier ---
province_files = sorted(glob.glob("moy_month/*_monthly_mean.nc"))
print(f"Nombre de fichiers province trouvés : {len(province_files)}")

# --- 3. Boucle sur chaque fichier province ---
for f in province_files:
    match = re.search(r"(\d{4})_(\d{2})_monthly_mean\.nc", f)
    if not match:
        print(f"Nom de fichier non reconnu, ignoré : {f}")
        continue
    year_num, month_num = int(match.group(1)), int(match.group(2))

    mask = (ds_sv["year"] == year_num) & (ds_sv["month"] == month_num)
    if not mask.any():
        continue

    sub = ds_sv.sel(time=mask)
    ds_prov = xr.open_dataset(f)

    prov_sel = ds_prov["prov"].sel(               # <-- ici : "prov" au lieu de "province"
        latitude=sub["latitude"],
        longitude=sub["longitude"],
        method="nearest"
    )

    province_values[mask.values] = prov_sel.values
    ds_prov.close()

# --- 4. Assigner la coordonnée province au dataset Sv ---
ds_sv = ds_sv.assign_coords(province=("time", province_values))

# --- 5. Vérification rapide ---
n_missing = np.sum(province_values == None) if province_values.dtype == object else np.isnan(province_values.astype(float)).sum()
print(f"Pings sans province assignée : {n_missing} / {ds_sv.dims['time']}")
print("Provinces uniques trouvées :", np.unique([p for p in province_values if p is not None]))